# Strange Places Dataset -- Exploration Notebook

**354,770 records of mysterious phenomena across 14 categories**

Author: [Luke Steuber](https://lukesteuber.com) | Bluesky: [@lukesteuber.com](https://bsky.app/profile/lukesteuber.com)

Dataset: [lukeslp/strange-places-dataset](https://huggingface.co/datasets/lukeslp/strange-places-dataset)

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from collections import Counter
import numpy as np

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Load data
with open('strange_places_v5.2.json') as f:
    raw = json.load(f)

df = pd.DataFrame(raw)
print(f"Loaded {len(df):,} records with {len(df.columns)} columns")
print(f"Columns: {list(df.columns)}")
print(f"\nDate range: {df['date'].dropna().min()} to {df['date'].dropna().max()}")
print(f"Categories: {df['category'].nunique()}")

## Category Distribution

How many records per phenomenon category?

In [ ]:
category_counts = df['category'].value_counts()

fig, ax = plt.subplots(figsize=(12, 7))
colors = plt.cm.Set3(np.linspace(0, 1, len(category_counts)))
bars = ax.barh(range(len(category_counts)), category_counts.values, color=colors)
ax.set_yticks(range(len(category_counts)))
ax.set_yticklabels([c.replace('_', ' ').title() for c in category_counts.index])
ax.set_xlabel('Number of Records')
ax.set_title('Strange Places -- Records by Category')
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

for bar, val in zip(bars, category_counts.values):
    ax.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nTotal records: {category_counts.sum():,}")

## Geographic Distribution

Where are these strange places located around the world?

In [ ]:
# Filter valid coordinates
geo = df.dropna(subset=['latitude', 'longitude'])
geo = geo[(geo['latitude'].between(-90, 90)) & (geo['longitude'].between(-180, 180))]

fig, ax = plt.subplots(figsize=(16, 8))

# Sample for performance -- plot up to 50k points per category
top_cats = geo['category'].value_counts().head(6).index
color_map = dict(zip(top_cats, plt.cm.tab10.colors[:6]))

for cat in top_cats:
    subset = geo[geo['category'] == cat].sample(n=min(15000, len(geo[geo['category'] == cat])), random_state=42)
    ax.scatter(subset['longitude'], subset['latitude'],
               s=0.5, alpha=0.3, label=cat.replace('_', ' ').title(),
               color=color_map[cat])

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Strange Places -- Global Distribution (Top 6 Categories)')
ax.legend(markerscale=10, loc='lower left', fontsize=9)
ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
plt.tight_layout()
plt.show()

print(f"Records with valid coordinates: {len(geo):,} / {len(df):,} ({100*len(geo)/len(df):.1f}%)")

## Temporal Distribution

When were these phenomena reported?

In [ ]:
# Parse dates
df['date_parsed'] = pd.to_datetime(df['date'], errors='coerce')
dated = df.dropna(subset=['date_parsed'])
dated = dated[dated['date_parsed'].dt.year >= 1900]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# By decade
dated['decade'] = (dated['date_parsed'].dt.year // 10) * 10
decade_counts = dated.groupby('decade').size()
axes[0].bar(decade_counts.index, decade_counts.values, width=8, color='#2196F3', edgecolor='white')
axes[0].set_xlabel('Decade')
axes[0].set_ylabel('Number of Records')
axes[0].set_title('Records by Decade (1900+)')
axes[0].yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# By month (seasonal patterns)
month_counts = dated['date_parsed'].dt.month.value_counts().sort_index()
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[1].bar(range(1, 13), [month_counts.get(i, 0) for i in range(1, 13)], color='#FF9800', edgecolor='white')
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(month_names)
axes[1].set_ylabel('Number of Records')
axes[1].set_title('Seasonal Distribution (All Years)')
axes[1].yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.show()

print(f"Records with valid dates (1900+): {len(dated):,}")

## Most Reported Locations

Which named places appear most frequently?

In [ ]:
# Top named locations
name_counts = df['name'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(range(len(name_counts)), name_counts.values, color='#9C27B0')
ax.set_yticks(range(len(name_counts)))
ax.set_yticklabels(name_counts.index)
ax.set_xlabel('Number of Records')
ax.set_title('Top 20 Most Frequently Named Locations')
ax.invert_yaxis()

for bar, val in zip(bars, name_counts.values):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## Country Distribution

Which countries have the most recorded strange places?

In [ ]:
# Country analysis (for records that have country data)
has_country = df.dropna(subset=['country'])
country_counts = has_country['country'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(range(len(country_counts)), country_counts.values, color='#00BCD4', edgecolor='white')
ax.set_xticks(range(len(country_counts)))
ax.set_xticklabels(country_counts.index, rotation=45, ha='right')
ax.set_ylabel('Number of Records')
ax.set_title('Top 15 Countries by Record Count')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

print(f"Records with country data: {len(has_country):,} / {len(df):,}")
print(f"Unique countries: {has_country['country'].nunique()}")

## Category Trends Over Time

How have different categories of reports changed across decades?

In [ ]:
# Category breakdown by decade
top_5_cats = df['category'].value_counts().head(5).index
time_df = dated[dated['category'].isin(top_5_cats)].copy()
time_df['decade'] = (time_df['date_parsed'].dt.year // 10) * 10
pivot = time_df.groupby(['decade', 'category']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 7))
pivot.plot(kind='bar', stacked=True, ax=ax, width=0.8,
           color=['#E91E63','#2196F3','#4CAF50','#FF9800','#9C27B0'])
ax.set_xlabel('Decade')
ax.set_ylabel('Number of Records')
ax.set_title('Top 5 Categories by Decade')
ax.legend([c.replace('_', ' ').title() for c in pivot.columns],
          loc='upper left', fontsize=9)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Meteorite Mass Distribution

For NASA meteorite records, what is the mass distribution?

In [ ]:
meteorites = df[df['category'] == 'nasa_meteorites'].dropna(subset=['mass_g'])
meteorites = meteorites[meteorites['mass_g'] > 0].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Log-scale histogram
axes[0].hist(np.log10(meteorites['mass_g']), bins=50, color='#FF5722', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Log10(Mass in grams)')
axes[0].set_ylabel('Count')
axes[0].set_title('Meteorite Mass Distribution (Log Scale)')

# Fall vs Found
fall_counts = df[df['category'] == 'nasa_meteorites']['fall_type'].value_counts()
axes[1].pie(fall_counts.values, labels=fall_counts.index, autopct='%1.1f%%',
            colors=['#3F51B5', '#FF9800'], startangle=90)
axes[1].set_title('Meteorites -- Fall vs Found')

plt.tight_layout()
plt.show()

print(f"Meteorites with mass data: {len(meteorites):,}")
print(f"Mass range: {meteorites['mass_g'].min():.1f}g to {meteorites['mass_g'].max():,.0f}g")
print(f"Median mass: {meteorites['mass_g'].median():,.1f}g")

## Summary Statistics

In [ ]:
print("=" * 60)
print("STRANGE PLACES DATASET -- SUMMARY")
print("=" * 60)
print(f"Total records:           {len(df):>12,}")
print(f"Categories:              {df['category'].nunique():>12}")
print(f"With coordinates:        {df.dropna(subset=['latitude','longitude']).shape[0]:>12,}")
print(f"With dates:              {df.dropna(subset=['date']).shape[0]:>12,}")
print(f"With country:            {df.dropna(subset=['country']).shape[0]:>12,}")
print(f"Unique countries:        {df['country'].dropna().nunique():>12}")
print(f"Date range:              {df['date'].dropna().min()} to {df['date'].dropna().max()}")
print()
print("Top 5 categories:")
for cat, count in df['category'].value_counts().head(5).items():
    pct = 100 * count / len(df)
    print(f"  {cat.replace('_',' ').title():<30} {count:>8,} ({pct:.1f}%)")
print("=" * 60)